# C2: LLM + RAG - Web Log Analysis

**Configuration:** LLM + RAG

## Objective

Add security knowledge retrieved from Chroma to the same LLM analysis. The prompt is standard; no detector or secure-prompt routing is used.

The four configurations use the same SQLite inputs, model, temperature, output
schema and record ordering. C2-C4 use the same retrieval query and top-k.
This notebook starts with four records for a low-cost demonstration; this is
not the complete thesis evaluation.

**Privacy and cost:** inference sends request fields to the model provider and
may incur charges. Use only data approved for external processing. Notebook
outputs and exported JSON can contain sensitive model responses.

## 1. Environment Setup

Select the project's Python environment and install its existing requirements
before opening the notebook. Set `OPENAI_API_KEY` in the environment or local
`.env` file. C2-C4 also require an initialized Chroma collection; see
`QUICKSTART.md`. No packages are installed by this notebook.

In [ ]:
import json
import os
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "database.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from inside the project repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from langchain_openai import ChatOpenAI
from src.config import MODEL_NAME
from src.database import MODEL_INPUT_FIELDS, retrieve_knowledge_documents
from src.experiments import (
    AnalysisReport, INJECTION_RULES, SECURE_SYSTEM, STANDARD_SYSTEM,
    analyze_record, detect_injection, load_experiment_records, retrieval_query,
)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the environment or the local .env file.")

CONFIGURATION = 'C2'
TEMPERATURE = 0.1
STRUCTURED_OUTPUT = True
RECOVER_MARKDOWN = True
llm = ChatOpenAI(model=MODEL_NAME, temperature=TEMPERATURE)
print("Configuration:", CONFIGURATION)
print("Model:", MODEL_NAME, "| Temperature:", TEMPERATURE)
print("Project:", PROJECT_ROOT)
print("Structured output:", STRUCTURED_OUTPUT, "| Markdown recovery:", RECOVER_MARKDOWN)

## 2. Data Models and Output Contract

The input consists of HTTP request fields only. Sample and pair IDs identify
results but are not included in the model prompt. Ground-truth annotations
remain in SQLite and are not used to select a route, build a query, or prompt
the model. The JSON schema below is validated after the response; a valid
schema does not establish that the classification is correct.

In [ ]:
print("Model input fields:", ", ".join(MODEL_INPUT_FIELDS))
print("\nOutput JSON schema:")
pprint(AnalysisReport.model_json_schema())

## 3. Load the Shared Experiment Dataset

Records are ordered by sample ID. Keep `N_RECORDS` and all settings equal
across notebooks for comparison; set it to `None` for the full dataset.
Four records are only a smoke test and may not represent every class or
injection variant. The loader raises an error if SQLite is missing or empty.

In [ ]:
N_RECORDS = 4
database_path = PROJECT_ROOT / "data" / "databases" / "experiments.sqlite3"
sample_logs = load_experiment_records(database_path, limit=N_RECORDS)
print("Records loaded:", len(sample_logs))
print("Sample / pair IDs:")
for log in sample_logs:
    print(" ", log["sample_id"], "/", log["pair_id"])

## 4. Prompt and Analysis Pipeline

The actual implementation is shared in `src/experiments.py`, so the four
notebooks use the same input formatting and JSON validation. The system prompt
is separate from the human message containing serialized references and HTTP
request data. The following cell shows the instructions actually used; prompt
separation is a mitigation, not a guarantee against prompt injection.

In [ ]:
print("Standard system prompt:\n", STANDARD_SYSTEM)

## 5. Knowledge Base and Retrieval

The same JSON serialization of model input fields is used as the retrieval
query in C2-C4. Chroma retrieves up to three documents using the same local
embedding function as ingestion. The model receives document IDs and content;
no ground-truth labels are included. Retrieval relevance is not guaranteed,
and must be evaluated separately.

In [ ]:
chroma_path = PROJECT_ROOT / "data" / "chroma_security_knowledge"
TOP_K = 3

def retrieve_evidence(query):
    return retrieve_knowledge_documents(query, chroma_path, limit=TOP_K)

print("Chroma path:", chroma_path)
print("Retrieval top-k:", TOP_K)
print("Query fields:", ", ".join(MODEL_INPUT_FIELDS))

### Inspect the Actual Implementation

The following cell prints the shared implementation used by this notebook.
Read it to follow input filtering, routing, message construction, retrieval,
schema validation, and result metadata. Not every branch applies to this
configuration; `CONFIGURATION` selects the relevant behavior.

In [ ]:
import inspect

print(inspect.getsource(analyze_record))

## 6. LLM + RAG Analyzer

`RAGAnalyzer` exposes the analysis step in the same class-based style as the
original notebooks. It delegates to `analyze_record`, which constructs the
messages, performs retrieval, calls the model, and validates
the JSON response. Provider and retrieval errors stop the run; invalid JSON is
preserved with an error rather than silently corrected.

Structured output asks the provider for the Pydantic JSON schema. The model
must support OpenAI Structured Outputs; unsupported models or API errors stop
the run. As a separate fallback, exactly one complete Markdown code block may
be unwrapped and validated. Both settings are identical across C1-C4 and can
be disabled in section 1 to reproduce the earlier behavior.

In [ ]:
class RAGAnalyzer:
    def __init__(self, model, retriever):
        self.model = model
        self.retriever = retriever

    def analyze(self, log):
        return analyze_record(
            CONFIGURATION, log, self.model, self.retriever,
            structured_output=STRUCTURED_OUTPUT, recover_markdown=RECOVER_MARKDOWN,
        )

analyzer = RAGAnalyzer(llm, retrieve_evidence)

## 7. Run Analysis

This cell makes one model call per record. It does not modify the SQLite
dataset. Avoid sharing outputs without checking them for sensitive content.

In [ ]:
results = []
for index, log in enumerate(sample_logs, start=1):
    print(f"Analyzing {index}/{len(sample_logs)}: {log['sample_id']}")
    results.append(analyzer.analyze(log))
print("Completed:", len(results))

## 8. Results Summary

Output validity measures adherence to the schema, not classification accuracy
or prompt-injection resistance. Routes and citation checks are operational
diagnostics, not attack-success metrics. Invalid responses remain available as
raw text and are not included in the classification distribution.

`raw_output_valid` measures direct schema validity; `markdown_recovered`
records successful recovery from a single Markdown block. `output_valid`
includes both. The raw response and any original validation error are retained.

In [ ]:
valid_count = sum(result["output_valid"] for result in results)
raw_valid_count = sum(result["raw_output_valid"] for result in results)
recovered_count = sum(result["markdown_recovered"] for result in results)
print(f"Valid directly: {raw_valid_count}/{len(results)}")
print(f"Recovered from Markdown: {recovered_count}/{len(results)}")
print(f"Valid overall: {valid_count}/{len(results)}")
print("Routes:", dict(Counter(result["route"] for result in results)))
print("Classifications:", dict(Counter(
    result["analysis"]["classification"]
    for result in results if result["output_valid"]
)))
for result in results:
    print("\nSample:", result["sample_id"], "| Route:", result["route"])
    print("  Retrieved IDs:", result["retrieved_evidence_ids"])
    if result["output_error"]:
        print("  Output error:", result["output_error"])
    if result["detector"] is not None:
        print("  Detector:", result["detector"])
    if result["citation_validation"] is not None:
        print("  Citation validation:", result["citation_validation"])

### Inspect a Report

Inspect the parsed analysis and original model response for one record. When
the output remains invalid, `analysis` is `None` and `output_error` explains the failure.
Review the evidence IDs and the response manually; a valid citation ID does
not verify factual support.

In [ ]:
example = results[0]
print("Sample:", example["sample_id"])
print("Parsed analysis:")
pprint(example["analysis"])
print("\nOriginal response:")
print(example["raw_response"])

## 9. Export Results

Results are saved as JSON, including sample IDs, the raw response and
diagnostics. Re-running overwrites the corresponding file; copy it elsewhere
first if you need to keep an earlier run. These notebooks do not yet compute
classification or attack-success metrics against ground-truth labels.

In [ ]:
output_dir = Path(os.getenv("EXPERIMENT_RESULTS_DIR", str(PROJECT_ROOT / "results")))
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / f"{CONFIGURATION.lower()}_results.json"
with output_file.open("w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=2)
print("Results saved to:", output_file)